# Idealista Barcelona Sales: URL Discovery + Detail Schema

Recommended workflow:

1. Paginate Idealista search-result pages to collect listing URLs.
2. Visit each listing URL once.
3. Extract details from structured schema/JSON-LD first, then fall back to visible page text.
4. Save checkpoints continuously so the run can be resumed.

This keeps the old-style columns, removes image fields, and adds `scraped_at`. It also includes `x` and `y`, where `x = longitude` and `y = latitude`.

In [21]:
# Run once if needed:
# %pip install selenium webdriver-manager beautifulsoup4 pandas tqdm lxml

import csv
import hashlib
import json
import random
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

In [22]:
START_URL = "https://www.idealista.com/en/venta-viviendas/barcelona-barcelona/"
SEARCH_SEED_URLS = [
    "https://www.idealista.com/en/venta-viviendas/barcelona/ciutat-vella/",
    "https://www.idealista.com/en/venta-viviendas/barcelona/eixample/",
    "https://www.idealista.com/en/venta-viviendas/barcelona/gracia/",
    "https://www.idealista.com/en/venta-viviendas/barcelona/horta-guinardo/",
    "https://www.idealista.com/en/venta-viviendas/barcelona/les-corts/",
    "https://www.idealista.com/en/venta-viviendas/barcelona/nou-barris/",
    "https://www.idealista.com/en/venta-viviendas/barcelona/sant-andreu/",
    "https://www.idealista.com/en/venta-viviendas/barcelona/sant-marti/",
    "https://www.idealista.com/en/venta-viviendas/barcelona/sants-montjuic/",
    "https://www.idealista.com/en/venta-viviendas/barcelona/sarria-sant-gervasi/",
]
OUT_DIR = Path("data")
OUT_DIR.mkdir(exist_ok=True)
HTML_CACHE_DIR = OUT_DIR / "html_cache"
SEARCH_CACHE_DIR = HTML_CACHE_DIR / "search_pages"
DETAIL_CACHE_DIR = HTML_CACHE_DIR / "detail_pages"
SEARCH_CACHE_DIR.mkdir(parents=True, exist_ok=True)
DETAIL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

URLS_CSV = OUT_DIR / "idealista_barcelona_sale_urls.csv"
OUTPUT_CSV = OUT_DIR / "idealista_barcelona_sale_properties_details.csv"

# Run controls. For your current state, set RUN_URL_DISCOVERY=False and RUN_DETAIL_SCRAPE=True.
RUN_URL_DISCOVERY = False
LOAD_URLS_FROM_CSV = True
RUN_DETAIL_SCRAPE = True
START_BROWSER = True

HEADLESS = False
CHROMEDRIVER_PATH = None   # set to r"C:\\path\\to\\chromedriver.exe" if Windows blocks downloaded drivers
USE_WEBDRIVER_MANAGER = True  # False uses Selenium Manager first; True uses webdriver-manager
USE_HTML_CACHE = True       # reuse cached HTML pages on reruns
REFRESH_HTML_CACHE = False  # set True if you want to re-fetch every page
WAIT_ON_BLOCK = True        # pause so you can solve CAPTCHA/block pages in the browser
MANUAL_UNBLOCK_CONFIRM = True  # ask you to press Enter after the page looks normal
BLOCK_WAIT_SECONDS = 300    # max time to wait for manual unblock before raising an error
STOP_ON_HARD_BLOCK = True   # stop immediately on Idealista's hard access-block page
MIN_DELAY = 2.5
MAX_DELAY = 6.5
MAX_RESULT_PAGES = None  # set to 2 for testing each seed URL; None means all detected pages
SEARCH_PAGE_COUNT = "until_empty"  # try pagina-N until no new listings; set an int, "auto", or None
MAX_PAGES_PER_SEED = 150  # safety cap for "until_empty" mode
STOP_AFTER_EMPTY_PAGES = 1
MAX_DETAILS = None       # set to 25 for testing; None means scrape all collected URLs
LOG_EVERY_DETAIL = 1     # set to 10 or 25 for quieter long runs

COLUMNS = [
    "propertyCode",
    "Link",
    "district",
    "neighborhood",
    "price",
    "size",
    "bed",
    "br",
    "floor",
    "address",
    "latitude",
    "longitude",
    "x",
    "y",
    "url",
    "description",
    "scraped_at",
]

In [23]:
def now_utc_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def log(message):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


def sleep():
    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))


def clean_text(value):
    if value is None:
        return None
    value = re.sub(r"\s+", " ", str(value)).strip()
    return value or None


def property_code(url):
    match = re.search(r"/inmueble/(\d+)/", str(url))
    return match.group(1) if match else None


def as_number(value):
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return value
    match = re.search(r"-?\d+(?:[.,]\d+)?", str(value).replace(".", ""))
    if not match:
        return None
    text = match.group(0).replace(",", ".")
    number = float(text)
    return int(number) if number.is_integer() else number


def setup_driver():
    log("Starting Chrome WebDriver")
    options = Options()
    options.add_argument("--window-size=1400,1000")
    options.add_argument("--lang=en-US,en")
    options.add_argument("--disable-blink-features=AutomationControlled")
    if HEADLESS:
        options.add_argument("--headless=new")

    if CHROMEDRIVER_PATH:
        log(f"Using explicit ChromeDriver path: {CHROMEDRIVER_PATH}")
        driver = webdriver.Chrome(service=Service(CHROMEDRIVER_PATH), options=options)
        driver.set_page_load_timeout(45)
        log("Chrome WebDriver ready")
        return driver

    if USE_WEBDRIVER_MANAGER:
        log("Resolving ChromeDriver with webdriver-manager")
        driver_path = ChromeDriverManager().install()
        log(f"ChromeDriver resolved: {driver_path}")
        try:
            driver = webdriver.Chrome(service=Service(driver_path), options=options)
        except OSError as exc:
            log(f"Windows blocked ChromeDriver at {driver_path}: {exc}")
            raise RuntimeError("Windows Application Control blocked ChromeDriver. Set CHROMEDRIVER_PATH to an approved chromedriver.exe path, or ask IT/Windows Security to allow this driver.") from exc
    else:
        try:
            log("Resolving ChromeDriver with Selenium Manager")
            driver = webdriver.Chrome(options=options)
        except (WebDriverException, OSError) as exc:
            log(f"Selenium Manager failed: {exc}")
            log("Falling back to webdriver-manager")
            driver_path = ChromeDriverManager().install()
            log(f"ChromeDriver resolved: {driver_path}")
            try:
                driver = webdriver.Chrome(service=Service(driver_path), options=options)
            except OSError as fallback_exc:
                log(f"Windows blocked ChromeDriver at {driver_path}: {fallback_exc}")
                raise RuntimeError("Windows Application Control blocked ChromeDriver. Set CHROMEDRIVER_PATH to an approved chromedriver.exe path, or ask IT/Windows Security to allow this driver.") from fallback_exc

    driver.set_page_load_timeout(45)
    log("Chrome WebDriver ready")
    return driver


def safe_get(driver, url):
    try:
        log(f"Loading: {url}")
        driver.get(url)
        return True
    except TimeoutException:
        log(f"Timeout while loading, keeping partial page: {url}")
        try:
            driver.execute_script("window.stop();")
        except WebDriverException:
            pass
        return True
    except WebDriverException as exc:
        log(f"Could not load {url}: {exc}")
        return False


def accept_cookies(driver):
    for label in ["Accept all", "Accept", "Agree", "Aceptar todas", "Aceptar"]:
        try:
            buttons = driver.find_elements(By.XPATH, f"//button[contains(normalize-space(.), '{label}')]")
            for button in buttons:
                if button.is_displayed() and button.is_enabled():
                    log("Accepting cookie banner")
                    button.click()
                    time.sleep(1)
                    return
        except WebDriverException:
            pass


def blocked(driver):
    try:
        body_text = driver.find_element(By.TAG_NAME, "body").text.lower()
        title = (driver.title or "").lower()
        current_url = (driver.current_url or "").lower()
    except WebDriverException as exc:
        log(f"Could not inspect visible page text for block check: {exc}")
        return False

    visible_text = " ".join([title, current_url, body_text])
    hard_markers = ["unusual traffic", "verify you are human", "access denied", "checking your browser"]
    if any(marker in visible_text for marker in hard_markers):
        return True

    # The word captcha can appear in normal page scripts, so only treat it as a block
    # when the visible page has little/no listing content.
    has_listing_content = "/inmueble/" in body_text or "property for sale" in body_text or "houses and flats" in body_text
    return "captcha" in visible_text and not has_listing_content


def hard_blocked(driver):
    try:
        body_text = driver.find_element(By.TAG_NAME, "body").text.lower()
    except WebDriverException as exc:
        log(f"Could not inspect visible page text for hard-block check: {exc}")
        return False
    hard_markers = [
        "se ha detectado un uso indebido",
        "el acceso se ha bloqueado",
        "póngase en contacto con el servicio de asistencia",
        "pongase en contacto con el servicio de asistencia",
    ]
    return any(marker in body_text for marker in hard_markers)


def wait_for_manual_unblock(driver, url):
    if not WAIT_ON_BLOCK:
        return False
    if STOP_ON_HARD_BLOCK and hard_blocked(driver):
        raise RuntimeError("Idealista is showing a hard access block. Stop scraping for now; progress already saved in the CSV checkpoints.")

    log("Idealista appears to be showing a block/CAPTCHA page")
    log("Use the open Chrome window to complete the check, then leave it on the Idealista page")
    if MANUAL_UNBLOCK_CONFIRM:
        input("After the Idealista page looks normal in Chrome, press Enter here to continue: ")
        if not blocked(driver):
            log("Manual unblock confirmed; continuing")
            return True
        log("The visible page still looks blocked after manual confirmation; waiting automatically")

    deadline = time.time() + BLOCK_WAIT_SECONDS
    while time.time() < deadline:
        try:
            if not blocked(driver):
                log("Block/CAPTCHA appears cleared; continuing")
                return True
            log("Still blocked; waiting 10 seconds before checking again")
            time.sleep(10)
        except WebDriverException as exc:
            log(f"Still waiting for unblock; browser error: {exc}")
            time.sleep(10)

    log(f"Still blocked after {BLOCK_WAIT_SECONDS} seconds on {url}")
    return False


def cache_path(cache_dir, url, suffix="html"):
    digest = hashlib.sha1(url.encode("utf-8")).hexdigest()[:16]
    code = property_code(url)
    stem = f"{code}_{digest}" if code else digest
    return cache_dir / f"{stem}.{suffix}"


def load_cached_html(cache_dir, url):
    path = cache_path(cache_dir, url)
    if USE_HTML_CACHE and not REFRESH_HTML_CACHE and path.exists():
        log(f"Cache hit: {path}")
        return path.read_text(encoding="utf-8")
    return None


def save_cached_html(cache_dir, url, html):
    if not USE_HTML_CACHE or not html:
        return None
    path = cache_path(cache_dir, url)
    path.write_text(html, encoding="utf-8")
    log(f"Cached HTML: {path}")
    return path


def current_html(driver):
    try:
        return driver.page_source
    except WebDriverException as exc:
        log(f"driver.page_source failed; trying JavaScript HTML read: {exc}")
        try:
            return driver.execute_script("return document.documentElement.outerHTML")
        except WebDriverException as js_exc:
            log(f"JavaScript HTML read also failed: {js_exc}")
            return None


def get_page_html(driver, url, cache_dir, wait_selector=None):
    cached_html = load_cached_html(cache_dir, url)
    if cached_html is not None:
        return cached_html

    if not safe_get(driver, url):
        return None
    accept_cookies(driver)
    if STOP_ON_HARD_BLOCK and hard_blocked(driver):
        raise RuntimeError("Idealista is showing a hard access block. Stop scraping for now; progress already saved in the CSV checkpoints.")
    if blocked(driver):
        if not wait_for_manual_unblock(driver, url):
            raise RuntimeError("Idealista is still showing a block/CAPTCHA. Stop or try again later.")
        accept_cookies(driver)
        if blocked(driver):
            raise RuntimeError("Idealista is still showing a block/CAPTCHA after manual wait.")

    if wait_selector:
        try:
            WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CSS_SELECTOR, wait_selector)))
        except TimeoutException:
            log(f"Wait selector not found after 20s: {wait_selector} on {url}")

    html = current_html(driver)
    if not html:
        return None
    save_cached_html(cache_dir, url, html)
    return html

## 1. URL Discovery

If `data/idealista_barcelona_sale_urls.csv` already exists, set `RUN_URL_DISCOVERY = False` and `LOAD_URLS_FROM_CSV = True` in the controls cell. Then this cell will load the URL checkpoint instead of revisiting search pages.

In [24]:
def parse_search_page(html, page_url):
    soup = BeautifulSoup(html, "lxml")
    rows = []
    cards = soup.select("article.item")
    if not cards:
        cards = [a.find_parent(["article", "div", "section"]) or a for a in soup.select("a[href*='/inmueble/']")]
    log(f"Found {len(cards)} listing cards on search page")

    for card in cards:
        link = card.select_one("a.item-link[href*='/inmueble/'], a[href*='/inmueble/']") if hasattr(card, "select_one") else card
        if not link:
            continue
        url = urljoin(page_url, link.get("href")).split("?")[0]
        if "/inmueble/" not in url:
            continue

        details_text = " | ".join(x.get_text(" ", strip=True) for x in card.select("span.item-detail, .item-detail-char span, [class*='item-detail']"))
        rows.append({
            "propertyCode": property_code(url),
            "url": url,
            "address_search": clean_text(link.get_text(" ", strip=True)),
            "price_search": clean_text((card.select_one("span.item-price, .item-price") or {}).get_text(" ", strip=True)) if card.select_one("span.item-price, .item-price") else None,
            "details_search": clean_text(details_text),
            "description_search": clean_text((card.select_one(".item-description") or {}).get_text(" ", strip=True)) if card.select_one(".item-description") else None,
            "source_page": page_url,
            "scraped_at": now_utc_iso(),
        })

    next_link = soup.select_one("a[rel='next'][href]")
    if not next_link:
        for a in soup.select("a[href]"):
            label = clean_text(a.get_text(" ", strip=True)) or ""
            aria = clean_text(a.get("aria-label")) or ""
            if label.lower() in {"next", "siguiente"} or "next" in aria.lower() or "siguiente" in aria.lower():
                next_link = a
                break
    next_url = urljoin(page_url, next_link["href"]) if next_link else None
    if next_url:
        log(f"Next search page detected: {next_url}")
    else:
        log("No next search page detected")
    return rows, next_url


def detect_page_count(html):
    soup = BeautifulSoup(html, "lxml")
    text = clean_text(soup.get_text(" ", strip=True)) or ""
    total_match = re.search(r"View\s+([\d,.]+)\s+results", text, flags=re.I)
    if not total_match:
        total_match = re.search(r":\s*([\d,.]+)\s+houses and flats", text, flags=re.I)
    if not total_match:
        return None
    total_results = as_number(total_match.group(1))
    cards = soup.select("article.item")
    if not cards:
        cards = [a.find_parent(["article", "div", "section"]) or a for a in soup.select("a[href*='/inmueble/']")]
    per_page = len(cards)
    if not total_results or not per_page:
        return None
    return int((total_results + per_page - 1) // per_page)


def numbered_search_page_urls(start_url, page_count):
    if not page_count:
        return None
    base = start_url.rstrip("/")
    return [start_url] + [f"{base}/pagina-{page_n}.htm" for page_n in range(2, page_count + 1)]


def numbered_search_page_url(start_url, page_n):
    if page_n == 1:
        return start_url
    return f"{start_url.rstrip('/')}/pagina-{page_n}.htm"


def seed_label(seed_url):
    return seed_url.rstrip("/").split("/")[-1]


def collect_urls(driver):
    if URLS_CSV.exists():
        df = pd.read_csv(URLS_CSV)
        rows = df.to_dict("records")
        seen = set(df["url"].dropna().astype(str))
        log(f"Loaded {len(seen):,} existing URLs from {URLS_CSV}")
    else:
        rows = []
        seen = set()
        log("No existing URL checkpoint found; starting URL discovery from scratch")

    seed_urls = SEARCH_SEED_URLS or [START_URL]
    log(f"URL discovery seeds: {len(seed_urls)}")

    for seed_index, seed_url in enumerate(seed_urls, start=1):
        label = seed_label(seed_url)
        log(f"Seed {seed_index}/{len(seed_urls)} start: {label} | {seed_url}")
        first_html = get_page_html(driver, seed_url, SEARCH_CACHE_DIR, wait_selector="a[href*='/inmueble/']")
        if not first_html:
            log(f"Skipping seed with no HTML: {seed_url}")
            continue

        if SEARCH_PAGE_COUNT == "auto":
            page_count = detect_page_count(first_html) or 1
            log(f"Detected {page_count} search pages for {label}")
        elif SEARCH_PAGE_COUNT == "until_empty":
            page_count = MAX_PAGES_PER_SEED
            log(f"Using numbered pagination until empty for {label}; safety cap {MAX_PAGES_PER_SEED} pages")
        elif isinstance(SEARCH_PAGE_COUNT, int):
            page_count = SEARCH_PAGE_COUNT
            log(f"Using configured {page_count} search pages for {label}")
        else:
            page_count = None
            log(f"Using Next-link pagination for {label}")

        numbered_pages = numbered_search_page_urls(seed_url, page_count)
        page_url = seed_url
        visited_pages = set()
        seen_page_signatures = set()
        page_n = 0
        empty_pages = 0

        while page_url and page_url not in visited_pages:
            page_n += 1
            if MAX_RESULT_PAGES and page_n > MAX_RESULT_PAGES:
                log(f"Reached MAX_RESULT_PAGES={MAX_RESULT_PAGES} for {label}; moving to next seed")
                break
            visited_pages.add(page_url)
            log(f"Search page {page_n}/{page_count or '?'} start for {label}")

            if page_n == 1 and page_url == seed_url:
                page_html = first_html
            else:
                page_html = get_page_html(driver, page_url, SEARCH_CACHE_DIR, wait_selector="a[href*='/inmueble/']")
            if not page_html:
                log(f"Skipping search page with no HTML: {page_url}")
                break

            page_rows, next_url = parse_search_page(page_html, page_url)
            page_signature = tuple(sorted(row["url"] for row in page_rows))
            repeated_page = bool(page_signature) and page_signature in seen_page_signatures
            if page_signature:
                seen_page_signatures.add(page_signature)
            for row in page_rows:
                row["seed_url"] = seed_url
                row["seed_label"] = label
            new_rows = [r for r in page_rows if r["url"] not in seen]
            rows.extend(new_rows)
            seen.update(r["url"] for r in new_rows)
            if len(page_rows) == 0:
                empty_pages += 1
            else:
                empty_pages = 0

            pd.DataFrame(rows).drop_duplicates("url", keep="last").to_csv(URLS_CSV, index=False, encoding="utf-8-sig")
            log(f"{label} page {page_n}: parsed {len(page_rows)} listings, +{len(new_rows)} new, {len(seen):,} total; checkpoint saved to {URLS_CSV}")
            if SEARCH_PAGE_COUNT == "until_empty" and empty_pages >= STOP_AFTER_EMPTY_PAGES:
                log(f"Stopping {label}: {empty_pages} consecutive empty/no-new page(s)")
                break
            if SEARCH_PAGE_COUNT == "until_empty" and repeated_page:
                log(f"Stopping {label}: page {page_n} repeated a previous page signature")
                break

            if SEARCH_PAGE_COUNT == "until_empty":
                page_url = numbered_search_page_url(seed_url, page_n + 1) if page_n < MAX_PAGES_PER_SEED else None
            elif numbered_pages:
                page_url = numbered_pages[page_n] if page_n < len(numbered_pages) else None
            else:
                page_url = next_url
            sleep()

        log(f"Seed complete: {label}; cumulative unique URLs: {len(seen):,}")

    log(f"URL discovery complete: {len(seen):,} unique listing URLs")
    return pd.DataFrame(rows).drop_duplicates("url", keep="last")


def load_urls_checkpoint(path=URLS_CSV):
    if not path.exists():
        raise FileNotFoundError(f"URL checkpoint not found: {path}. Run URL discovery first or set URLS_CSV to the right file.")
    df = pd.read_csv(path)
    if "url" not in df.columns:
        raise ValueError(f"URL checkpoint must contain a 'url' column: {path}")
    df = df.dropna(subset=["url"]).drop_duplicates("url", keep="last")
    log(f"Loaded {len(df):,} URLs from checkpoint: {path}")
    return df


driver = setup_driver() if START_BROWSER else None
if RUN_URL_DISCOVERY:
    if driver is None:
        driver = setup_driver()
    urls_df = collect_urls(driver)
elif LOAD_URLS_FROM_CSV:
    urls_df = load_urls_checkpoint(URLS_CSV)
else:
    urls_df = pd.DataFrame(columns=["url", "propertyCode"])
    log("URL discovery skipped and LOAD_URLS_FROM_CSV=False; urls_df is empty")
urls_df.head(), urls_df.shape

[18:55:09] Starting Chrome WebDriver
[18:55:09] Resolving ChromeDriver with webdriver-manager
[18:55:12] ChromeDriver resolved: C:\Users\sffra\.wdm\drivers\chromedriver\win64\147.0.7727.57\chromedriver-win32/chromedriver.exe
[18:55:13] Chrome WebDriver ready
[18:55:13] Loaded 10,619 URLs from checkpoint: data\idealista_barcelona_sale_urls.csv


(   propertyCode                                               url  \
 0     109696793  https://www.idealista.com/en/inmueble/109696793/   
 1     110509586  https://www.idealista.com/en/inmueble/110509586/   
 2     110630744  https://www.idealista.com/en/inmueble/110630744/   
 3     110063350  https://www.idealista.com/en/inmueble/110063350/   
 4     110182478  https://www.idealista.com/en/inmueble/110182478/   
 
                                       address_search price_search  \
 0  Flat / apartment in Calle d'Aragó, La Dreta de...  1,199,000 €   
 1  Flat / apartment in Calle de Tamarit, Sant Ant...    545,000 €   
 2  Flat / apartment in Calle del Consell de Cent,...  1,390,000 €   
 3  Flat / apartment in Calle de Girona, La Dreta ...  2,200,000 €   
 4  Flat / apartment in Calle de Mallorca, La Dret...  2,195,000 €   
 
                                       details_search  \
 0  3 bed. 164 m² 1st floor exterior with lift | 3...   
 1  2 bed. 80 m² 4th floor exterior with l

## 2. Detail Scrape

This step starts from `urls_df`. If `urls_df` is not already loaded, it will load `data/idealista_barcelona_sale_urls.csv` directly. It checkpoints every completed property row to `data/idealista_barcelona_sale_properties_details.csv`.

In [25]:
def json_objects(value):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from json_objects(child)
    elif isinstance(value, list):
        for child in value:
            yield from json_objects(child)


def load_jsonld(soup):
    objects = []
    for script in soup.select("script[type='application/ld+json']"):
        raw = script.string or script.get_text()
        if not raw:
            continue
        try:
            objects.extend(list(json_objects(json.loads(raw))))
        except Exception:
            continue
    return objects


def first_schema_value(objects, keys):
    keys = {k.lower() for k in keys}
    for obj in objects:
        for key, value in obj.items():
            if str(key).lower() in keys and value not in (None, "", []):
                return value
    return None


def schema_address(objects):
    value = first_schema_value(objects, ["address"])
    if isinstance(value, dict):
        parts = [value.get(k) for k in ["streetAddress", "addressLocality", "addressRegion", "postalCode"]]
        return clean_text(", ".join(str(x) for x in parts if x))
    return clean_text(value)


def schema_geo(objects):
    for obj in objects:
        if "latitude" in obj and "longitude" in obj:
            return obj.get("latitude"), obj.get("longitude")
        geo = obj.get("geo")
        if isinstance(geo, dict) and "latitude" in geo and "longitude" in geo:
            return geo.get("latitude"), geo.get("longitude")
    return None, None


def regex_geo(html):
    patterns = [
        r'"latitude"\s*:\s*([\-\d.]+)\s*,\s*"longitude"\s*:\s*([\-\d.]+)',
        r'"longitude"\s*:\s*([\-\d.]+)\s*,\s*"latitude"\s*:\s*([\-\d.]+)',
        r'lat(?:itude)?["\']?\s*[:=]\s*["\']?([\-\d.]+).*?lon(?:gitude)?["\']?\s*[:=]\s*["\']?([\-\d.]+)',
    ]
    for i, pattern in enumerate(patterns):
        match = re.search(pattern, html, flags=re.I | re.S)
        if match and i == 1:
            return match.group(2), match.group(1)
        if match:
            return match.group(1), match.group(2)
    return None, None


def visible_text(soup, selectors):
    for selector in selectors:
        node = soup.select_one(selector)
        if node:
            text = clean_text(node.get_text(" ", strip=True))
            if text:
                return text
    return None


def detail_text(soup, fallback=None):
    chunks = [x.get_text(" ", strip=True) for x in soup.select(".details-property_features li, .info-features span, .details-property-feature-one, .details-property-feature-two, .item-detail")]
    text = clean_text(" | ".join(chunks))
    return text or fallback


def parse_features(text):
    text = clean_text(text) or ""
    out = {"size": None, "bed": None, "br": None, "floor": None}

    m = re.search(r"([\d.,]+)\s*m[²2]", text, flags=re.I)
    if m:
        out["size"] = as_number(m.group(1))

    m = re.search(r"(\d+)\s*bed", text, flags=re.I)
    if m:
        out["bed"] = int(m.group(1))

    m = re.search(r"(\d+)\s*bath", text, flags=re.I)
    if m:
        out["br"] = int(m.group(1))

    for pattern in [r"(\d+)(?:st|nd|rd|th)?\s*floor", r"floor\s*(\d+)", r"(ground floor|basement|semi-basement|mezzanine|top floor)"]:
        m = re.search(pattern, text, flags=re.I)
        if m:
            out["floor"] = m.group(1).lower()
            break
    return out


def parse_location_from_address(address):
    parts = [clean_text(x) for x in str(address or "").split(",")]
    parts = [x for x in parts if x]
    district = None
    neighborhood = None
    if len(parts) >= 3:
        neighborhood = parts[-2]
        district = parts[-1].replace("Barcelona", "").strip() or None
    elif len(parts) == 2:
        neighborhood = parts[-1]
    return district, neighborhood


def parse_listing_detail(html, url, search_row):
    soup = BeautifulSoup(html, "lxml")
    objects = load_jsonld(soup)
    log(f"Parsing detail page for {search_row.get('propertyCode') or property_code(url)}: {len(objects)} schema objects found")
    features = parse_features(detail_text(soup, search_row.get("details_search")))

    schema_lat, schema_lon = schema_geo(objects)
    regex_lat, regex_lon = regex_geo(html)
    latitude = schema_lat or regex_lat
    longitude = schema_lon or regex_lon

    address = (
        schema_address(objects)
        or visible_text(soup, ["span.main-info__title-main", ".main-info__title-main", "h1"])
        or search_row.get("address_search")
    )
    district, neighborhood = parse_location_from_address(address)

    price_value = first_schema_value(objects, ["price"])
    price = clean_text(price_value) or visible_text(soup, ["span.info-data-price", ".info-data-price", "span.item-price", ".item-price"])

    description = (
        clean_text(first_schema_value(objects, ["description"]))
        or visible_text(soup, ["div.comment", ".adCommentsLanguage", "#details .comment", "[class*='description']"])
        or search_row.get("description_search")
    )

    size_schema = first_schema_value(objects, ["floorSize", "size", "area"])
    if isinstance(size_schema, dict):
        size_schema = size_schema.get("value") or size_schema.get("amount")

    return {
        "propertyCode": search_row.get("propertyCode") or property_code(url),
        "Link": "LINK",
        "district": district,
        "neighborhood": neighborhood,
        "price": price or search_row.get("price_search"),
        "size": as_number(size_schema) or features["size"],
        "bed": as_number(first_schema_value(objects, ["numberOfBedrooms", "numberOfRooms"])) or features["bed"],
        "br": as_number(first_schema_value(objects, ["numberOfBathroomsTotal", "numberOfBathrooms"])) or features["br"],
        "floor": clean_text(first_schema_value(objects, ["floorLevel", "floor"])) or features["floor"],
        "address": address,
        "latitude": latitude,
        "longitude": longitude,
        "x": longitude,
        "y": latitude,
        "url": url,
        "description": description,
        "scraped_at": now_utc_iso(),
    }

In [ ]:
def load_existing_output():
    if OUTPUT_CSV.exists():
        df = pd.read_csv(OUTPUT_CSV)
        log(f"Loaded {len(df):,} existing detail rows from {OUTPUT_CSV}")
        return df.reindex(columns=COLUMNS), set(df["url"].dropna().astype(str))
    log("No existing detail checkpoint found; starting detail scrape from scratch")
    return pd.DataFrame(columns=COLUMNS), set()


def summarize_detail_row(row):
    fields = ["price", "size", "bed", "br", "floor", "address", "latitude", "longitude", "description"]
    present = [name for name in fields if row.get(name) not in (None, "")]
    missing = [name for name in fields if row.get(name) in (None, "")]
    return f"present={present}; missing={missing}"


def scrape_details(driver, urls_df):
    existing, done = load_existing_output()
    rows = existing.to_dict("records")
    todo = urls_df[~urls_df["url"].astype(str).isin(done)].copy()
    if MAX_DETAILS:
        todo = todo.head(MAX_DETAILS)

    log(f"Detail pages to scrape: {len(todo):,}; already done: {len(done):,}")
    for index, search_row in enumerate(tqdm(todo.to_dict("records"), desc="Detail pages"), start=1):
        url = search_row["url"]
        code = search_row.get("propertyCode") or property_code(url)
        if LOG_EVERY_DETAIL and (index == 1 or index % LOG_EVERY_DETAIL == 0):
            log(f"Detail {index}/{len(todo)} start: {code} | {url}")
        page_html = get_page_html(driver, url, DETAIL_CACHE_DIR, wait_selector="body")
        if not page_html:
            log(f"Skipping detail page with no HTML: {url}")
            continue

        parsed = parse_listing_detail(page_html, url, search_row)
        rows.append(parsed)
        pd.DataFrame(rows).reindex(columns=COLUMNS).drop_duplicates("url", keep="last").to_csv(
            OUTPUT_CSV, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_MINIMAL
        )
        if LOG_EVERY_DETAIL and (index == 1 or index % LOG_EVERY_DETAIL == 0):
            log(f"Detail {index}/{len(todo)} saved: {code}; {summarize_detail_row(parsed)}")
        sleep()

    final_df = pd.DataFrame(rows).reindex(columns=COLUMNS).drop_duplicates("url", keep="last")
    log(f"Detail scrape complete: {len(final_df):,} rows saved to {OUTPUT_CSV}")
    return final_df


if RUN_DETAIL_SCRAPE:
    if "urls_df" not in globals() or urls_df.empty:
        urls_df = load_urls_checkpoint(URLS_CSV)
    if driver is None:
        driver = setup_driver()
    properties_df = scrape_details(driver, urls_df)
else:
    log("Detail scrape skipped because RUN_DETAIL_SCRAPE=False")
    properties_df, _ = load_existing_output()
properties_df.head(), properties_df.shape

[18:55:14] Loaded 18 existing detail rows from data\idealista_barcelona_sale_properties_details.csv
[18:55:14] Detail pages to scrape: 10,601; already done: 18


Detail pages:   0%|          | 0/10601 [00:00<?, ?it/s]

[18:55:16] Detail 1/10601 start: 110699054 | https://www.idealista.com/inmueble/110699054/
[18:55:16] Loading: https://www.idealista.com/inmueble/110699054/
[18:55:21] Cached HTML: data\html_cache\detail_pages\110699054_2f835a612d13a9da.html
[18:55:21] Parsing detail page for 110699054: 0 schema objects found
[18:55:21] Detail 1/10601 saved: 110699054; present=['price', 'address', 'description']; missing=['size', 'bed', 'br', 'floor', 'latitude', 'longitude']


Detail pages:   0%|          | 1/10601 [00:08<24:59:34,  8.49s/it]

[18:55:24] Detail 2/10601 start: 108917125 | https://www.idealista.com/inmueble/108917125/
[18:55:24] Loading: https://www.idealista.com/inmueble/108917125/
[18:55:25] Cached HTML: data\html_cache\detail_pages\108917125_87fc3bbe65d09be8.html
[18:55:25] Parsing detail page for 108917125: 0 schema objects found
[18:55:25] Detail 2/10601 saved: 108917125; present=['price', 'address', 'description']; missing=['size', 'bed', 'br', 'floor', 'latitude', 'longitude']


...

[18:55:33] Detail 592/10601 start: 106687670 | https://www.idealista.com/en/inmueble/106687670/
[18:55:33] Loading: https://www.idealista.com/en/inmueble/106687670/
[18:55:33] Could not load https://www.idealista.com/en/inmueble/106687670/: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	chromedriver!GetHandleVerifier [0xf0c733+10b73]
	chromedriver!GetHandleVerifier [0xf0c864+10ca4]
	chromedriver!(No symbol) [0xce1f3e]
	chromedriver!(No symbol) [0xd1f5c5]
	chromedriver!(No symbol) [0xd4dad6]
	chromedriver!(No symbol) [0xd48f42]
	chromedriver!(No symbol) [0xd48561]
	chromedriver!(No symbol) [0xcb531e]
	chromedriver!(No symbol) [0xcb58be]
	chromedriver!(No symbol) [0xcb5d9d]
	chromedriver!GetHandleVerifier [0x116fe04+274244]
	chromedriver!GetHandleVerifier [0x116b459+26f899]
	chromedriver!GetHandleVerifier [0x1189bd5+28e015]
	chromedriver!GetHandleVer

Detail pages:   6%|▌         | 621/10601 [00:17<00:41, 239.89it/s]

[18:55:33] Detail 622/10601 start: 110462241 | https://www.idealista.com/en/inmueble/110462241/
[18:55:33] Loading: https://www.idealista.com/en/inmueble/110462241/
[18:55:33] Could not load https://www.idealista.com/en/inmueble/110462241/: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	chromedriver!GetHandleVerifier [0xf0c733+10b73]
	chromedriver!GetHandleVerifier [0xf0c864+10ca4]
	chromedriver!(No symbol) [0xce1f3e]
	chromedriver!(No symbol) [0xd1f5c5]
	chromedriver!(No symbol) [0xd4dad6]
	chromedriver!(No symbol) [0xd48f42]
	chromedriver!(No symbol) [0xd48561]
	chromedriver!(No symbol) [0xcb531e]
	chromedriver!(No symbol) [0xcb58be]
	chromedriver!(No symbol) [0xcb5d9d]
	chromedriver!GetHandleVerifier [0x116fe04+274244]
	chromedriver!GetHandleVerifier [0x116b459+26f899]
	chromedriver!GetHandleVerifier [0x1189bd5+28e015]
	chromedriver!GetHandleVer

Detail pages:   6%|▌         | 646/10601 [00:17<00:43, 228.70it/s]

[18:55:33] Detail 647/10601 start: 105217404 | https://www.idealista.com/en/inmueble/105217404/
[18:55:33] Loading: https://www.idealista.com/en/inmueble/105217404/
[18:55:33] Could not load https://www.idealista.com/en/inmueble/105217404/: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	chromedriver!GetHandleVerifier [0xf0c733+10b73]
	chromedriver!GetHandleVerifier [0xf0c864+10ca4]
	chromedriver!(No symbol) [0xce1f3e]
	chromedriver!(No symbol) [0xd1f5c5]
	chromedriver!(No symbol) [0xd4dad6]
	chromedriver!(No symbol) [0xd48f42]
	chromedriver!(No symbol) [0xd48561]
	chromedriver!(No symbol) [0xcb531e]
	chromedriver!(No symbol) [0xcb58be]
	chromedriver!(No symbol) [0xcb5d9d]
	chromedriver!GetHandleVerifier [0x116fe04+274244]
	chromedriver!GetHandleVerifier [0x116b459+26f899]
	chromedriver!GetHandleVerifier [0x1189bd5+28e015]
	chromedriver!GetHandleVer

## 3. Quality checks

In [ ]:
properties_df = pd.read_csv(OUTPUT_CSV)
display(properties_df.head())
display(properties_df.isna().mean().sort_values(ascending=False).to_frame("missing_share"))
print(f"Rows: {len(properties_df):,}")
print(f"Unique property codes: {properties_df['propertyCode'].nunique():,}")
print(f"Rows with x/y: {properties_df[['x', 'y']].notna().all(axis=1).sum():,}")
print(f"Output: {OUTPUT_CSV.resolve()}")

In [ ]:
driver.quit()